In [9]:
import os
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns',None)

In [2]:
dta_dir = 'S:/LLC_0028/data/processed_study_data/'

In [5]:
%%time 
for f in os.listdir(dta_dir):
    
    if 'harmonise' in f:
        
        #study=f.split('_')[3]
        continue
        
    else:
        study=f.split('_')[2][:-3]
        
    print(study)
    
    if study=='alspacm':
        study='alspac_m'
    elif study=='alspacy':
        study='alspac_y'
    else:
        study=study
    
    df = pd.read_csv(dta_dir + f)
    
    #read in dictionary
    with open('llc_0028_symplkup_syntax_v1.json','r') as f:
        symp2harmonised = json.load(f)
    
    symp2harmonised = symp2harmonised[study] 
    
    #map columns to harmonised symptoms
    df = df.rename(columns=symp2harmonised)
    
    #encode text vars
    df = df.replace('No',0)
    df = df.replace('Yes',1)
    
    df = df.replace(2,0)
    
    #if someone didnt answer a questionnaire, all symptoms are nan
    #drop these participants
    symptoms = symp2harmonised.values()
    
    df = df.dropna(axis=0, how='all', subset=symptoms)
    
    #in each study, only have columns where symptom was asked about
    #at study-level, nan = absent
    df = df.replace(np.nan,0)
    
    #missing so replace with nan
    df = df.replace(999914,np.nan)
    df = df.replace(999906,np.nan)
    df = df.replace(-99,np.nan)

    df = df.dropna(axis=0, how='all', subset=symptoms)
    
    to_keep = list(symp2harmonised.values()) + [c for c in df.columns if '_0028_' in c]

    df = df[to_keep]

    #merge duplicate columns and create harmonised dataset
    harmonised_df = pd.DataFrame()

    for c in df.columns.unique(): #loop over unique columns

        if len(df[c].shape)>1:#if more than one column with same name

            #sum columns, ignoring NaN  
            total = df[c].sum(axis=1, skipna=True, min_count=1)
            #ensure maximum of column is 1, and add to harmonised df 
            harmonised_df[c] = [min(v,1) if not np.isnan(v) else v for v in total]

        else:

            harmonised_df[c] = df[c]

    #save harmonised data
    study = ('').join(study.split('_'))


    harmonised_df.to_csv(f'S:/LLC_0028/data/harmonised_symptom_data/harmonised_symptom/llc_0028_{study}hs_data_v3.csv')

alspacm
alspacy
bcs70
bib
mcs
ncds
nextstep
nhsd46
sabre
track19
twins
Wall time: 2.37 s
